Name

Labpartner(s)

In [1]:
#import statements go here


# Class 10.2

Today we are having a guest lecture by Kesse Asante on species distribution modeling in R. 

**W.1** Practice python #13

The Fibonnaci seqence is a sequence of numbers where the next number in the sequence is the sum of the previous two numbers in the sequence. The sequence looks like this: 1, 1, 2, 3, 5, 8, 13, …

Write a program that asks the user how many Fibonnaci numbers to generate and then generates them. Take this opportunity to think about how you can use functions. Make sure to ask the user to enter the number of numbers in the sequence to generate.

In [4]:
def fibonacci():
    num = int(input("enter number:"))
    i = 1
    fib1 = [1]
    fib2 = [1,1]
    if num == 1:
        fib = fib1
    elif num == 2:
        fib = fib2
    elif num > 2:
        fib = [1,1]
        while i < (num - 1):
            fib.append(fib[i] + fib[i-1])
            i += 1
    return fib
print(fibonacci())

enter number: 52


[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418, 317811, 514229, 832040, 1346269, 2178309, 3524578, 5702887, 9227465, 14930352, 24157817, 39088169, 63245986, 102334155, 165580141, 267914296, 433494437, 701408733, 1134903170, 1836311903, 2971215073, 4807526976, 7778742049, 12586269025, 20365011074, 32951280099]


# Lecture 10.2

### Agenda: Asante will have you install some R, then he will walk you through some species distribution modeling

**Please see the pdf document for Lab 10.2 and get the downloads going.**

### Questions

# Lab 10.2

**E.0** Complete the previous labs if you are behind.

**E.1** Asante will give you some exercises to do in R

Developing species distribution models in R: computer moels to establish relationshups between speciies occurence and environmental conditions controling the distribution 
-->look at ebvironmental tolerece, habitate preference, spacial predications 
use species records and enivornmental data: what are the connections 

model building: 
concepetrualizes: what environmental layers are important for this species 
data preparation: environmental and spcies records 
model fitting: different models fit to different algorithiums 
assessment: whther the model is working or not...spesitivity, area, TSS
Predictions: give the model future data to predict future species distribution 

physical, hemical, and biological variables to ontrol species distributions 



**E.2** Make notes for yourself on progamming tecniques and commands you learned in the this week, including examples, comments and explainitory text. You can do this here or in a separate notebook that you link to here. Basically, you are making a cheat sheet for yourself.

In [ ]:
#script from R for HW:

##packages installed here? Some of them were installed below tho.  
library(terra)
library(bDSSDMTools)
library(ggplot2)
library(sf)

##Seagrass example first 
landmass <- vect("Sci_Comp26/Data/Landmass/landmass.shp")
plot(landmass, col = "grey", border = "black", axes = TRUE)

readLines("Sci_Comp26/Data/Seagrass/species_presence_absence_records.csv", n =3)
records <- read.csv("Sci_Comp26/Data/Seagrass/species_presence_absence_records.csv", sep = ";")
#transform dataframe into a spacial object
records <- vect(records, geom = c("Lon", "Lat"))

#plot the lanmass and data
plot(landmass, main = "Species distribution records", col = "#D4D4D4", border = "#D4D4D4", axes = TRUE)
plot(records, y = "PA", col = c("#c29431", "#043259"), pch = 16, cex = 0.5, add = TRUE)

#importenviornmental data
files <- c("Sci_Comp26/Data/Raster_data/Present Benthic DissolvedMolecularOxygen Mean.tif", # Oxygen
           "Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMax.tif", # Temperature Max
           "Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMin.tif", # Temperature Min
           "Sci_Comp26/Data/Raster_data/Present Benthic pH Mean.tif", # pH
           "Sci_Comp26/Data/Raster_data/Present Benthic TotalPrimaryProductionPhyto Mean.tif")

#look at all enviornmental data
environmentalLayers <- rast(files)
plot(environmentalLayers)

#zoom into sudy region only
myStudyRegion <- studyRegion(rasterLayers=environmentalLayers,
                             presences=records, # presence records
                             distanceThreshold=200000, # distance from records in meters
                             bathymetryLayer = "Sci_Comp26/Data/Raster_data/Bathymetry.tif", # bathymetry layer
                             depthPref=c(0,-500), # known vertical distribution
                             intertidalLayer = "Sci_Comp26/Data/Raster_data/CoastLine.tif" ) # consider intertidal region
plot(myStudyRegion, main="Study region", col="#979797", axes=TRUE)


#crop enviornmental layers to look at only study region area
environmentalLayers <- crop(environmentalLayers, myStudyRegion)
environmentalLayers <- mask(environmentalLayers, myStudyRegion)
plot(environmentalLayers, axes = TRUE)

#extract enviormental values and make an object with all info to model; fit a binomial GLM to the staset
modelData <- generateModelData(records, envConditions = environmentalLayers)
model <- trainModel(modelData, algorithm = "brt")
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, main = "Predicted species distribution", col = rev(topo.colors(100)))

#Generating pseudo-absence data for a brt model
presences <- read.csv("Sci_Comp26/Data/Corals/occurrenceRecords.csv")
presences <-vect(presences, geom = c("Lon", "Lat"))
plot(presences, col = "green", pch = 16, cex = 0.4)
plot(landmass, main  = "Coral Species Records", col = "grey", border = "grey", axes = TRUE)
plot(presences, col = "green", pch = 16, cex = 0.4, add = TRUE)

#environmental data
files <- c("Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMax.tif", "Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMin.tif","Sci_Comp26/Data/Raster_data/Present Benthic pH Mean.tif","Sci_Comp26/Data/Raster_data/Present Benthic Salinity Mean.tif","Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMin.tif","Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMax.tif")
environmentalLayers <- rast(files)
plot(environmentalLayers)

#zoom into study region
ct_extent <- ext(95, 160, -12, 15)
presences_ct <- crop(presences, ct_extent)
plot(presences_ct)
plot(landmass, main  = "Coral Species Records", col = "grey", border = "grey", axes = TRUE)
plot(presences_ct, col = "green", pch = 16, cex = 0.4, add = TRUE)
myStudyRegion <- studyRegion(rasterLayers = environmentalLayers, presences = presences_ct, distanceThreshold = 200000, bathymetryLayer= "Sci_Comp26/Data/Raster_data/Bathymetry.tif", depthPref = c(0, -60))
plot(myStudyRegion, main = "Study Region", col = "grey", axes = TRUE)

#crop envir layers to look at only study region
environmentalLayers <- crop(environmentalLayers, myStudyRegion)
environmentalLayers <- mask(environmentalLayers, myStudyRegion)
plot(environmentalLayers, axes = TRUE)

#generate pseudo-absence data
absences <- pseudoAbsences(rasterLayers = environmentalLayers, presences = presences_ct, n = 2000)
#zoom into study region
landmassCT <-crop(landmass, myStudyRegion)
plot(landmass, main  = "Coral Triangle Species records", col = "grey", border = "grey", axes = TRUE)
plot(absences, col ="gold", pch = 16, cex = 0.5, add = TRUE)
plot(presences_ct, col = "blue", pch = 16, cex = 0.5, add = TRUE)

#combine the presence adn absence data to model
presences_ct$PA <-1
absences$PA <- 0 
records <- rbind(presences_ct, absences)
plot(landmassCT, main = "Coral Reefs Species records", col = "grey", border = "grey", axes = TRUE)
plot(records, y = "PA", col = c("gold", "blue"), pch = 16, cex = 0.5, add= TRUE)

#extract envi alues and make an object with all info to model
modelData <- generateModelData(records, envConditions = environmentalLayers, method = "blocks", kFolds = 10)
modelData$plotDatasets

#fitting BRT model to dataset
model <- trainModel(modelData, algorithm = "brt")
#predict the distribution with the model over the raster stack?
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, main = "Predicted Species distribution", col = rev(topo.colors(100)))


#generating background data for a maxent model
presences <- read.csv("Sci_Comp26/Data/Corals/occurrenceRecords.csv")
presences <-vect(presences, geom = c("Lon", "Lat"))
plot(presences, col = "green", pch = 16, cex = 0.4)

#generate background points
background <- backgroundInformation(rasterLayers = environmentalLayers, n = 10000)
plot(landmassCT, main = "Colar triangle species records", col = "grey", border = "grey", axes = TRUE)
plot(background, col = "gold", pch = 16, cex = 0.5, add= TRUE)
plot(presences_ct, col = "blue", pch = 16, cex = 0.5, add = TRUE)

#combine presences andbackgorunf inta a unique object for modeling 
presences_ct$PA <- 1
background$PA <- 0 
records <- rbind(background, presences_ct)

plot(landmassCT, main = "coral reefs species records", col = "grey", border = "grey", axes = TRUE)
plot(records, y = "PA", col = c("gold", "blue"), pch = 16, cex = 0.5, add= TRUE)

#Extract evni values and make an object with all info to model
modelData <- generateModelData(records, envConditions = environmentalLayers, method = "blocks", kFolds = 10)
modelData$plotDatasets

#this won't work because we hav to rename the files 
model <- trainModel(modelData, algorithm = "maxent")

#rename the files to remove spaces
names(environmentalLayers)
names(environmentalLayers) <- c("Present_Benthic_OceanTemperature_LtMax","Present_Benthic_OceanTemperature_LtMin","Present_Benthic_pH_Mean","Present_Benthic_Salinity_Mean","Present_Surface_OceanTemperature_LtMax","Present_Surface_OceanTemperature_LtMin.tif")
plot(environmentalLayers, axes = TRUE)
#rerune the generation line
modelData <- generateModelData(records, envConditions = environmentalLayers, method = "blocks", kFolds = 10)
#now it works 
model <- trainModel(modelData, algorithm = "maxent")
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, main  = "predicted Species distribution", col = rev(topo.colors(100)))


#do the same thingsfor a new study region 

presences <- read.csv("Sci_Comp26/Data/Corals/occurrenceRecords.csv")
presences <-vect(presences, geom = c("Lon", "Lat"))
plot(presences, col = "green", pch = 16, cex = 0.4)
plot(landmass, main  = "Coral Species Records", col = "grey", border = "grey", axes = TRUE)
plot(presences, col = "green", pch = 16, cex = 0.4, add = TRUE)

files <- c("Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMax.tif", "Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMin.tif","Sci_Comp26/Data/Raster_data/Present Benthic pH Mean.tif","Sci_Comp26/Data/Raster_data/Present Benthic Salinity Mean.tif","Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMin.tif","Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMax.tif")
environmentalLayers <- rast(files)
plot(environmentalLayers)

ct_extent <- ext(25, 47, 3, 30)
presences_ct <- crop(presences, ct_extent)
plot(presences_ct)
plot(landmass, main  = "Coral Species Records", col = "grey", border = "grey", axes = TRUE)
plot(presences_ct, col = "green", pch = 16, cex = 0.4, add = TRUE)

myStudyRegion <- studyRegion(rasterLayers = environmentalLayers, presences = presences_ct, distanceThreshold = 200000, bathymetryLayer= "Sci_Comp26/Data/Raster_data/Bathymetry.tif", depthPref = c(0, -60))
plot(myStudyRegion, main = "Study Region", col = "grey", axes = TRUE)

environmentalLayers <- crop(environmentalLayers, myStudyRegion)
environmentalLayers <- mask(environmentalLayers, myStudyRegion)
plot(environmentalLayers, axes = TRUE)


absences <- pseudoAbsences(rasterLayers = environmentalLayers, presences = presences_ct, n = 2000)
landmassCT <-crop(landmass, myStudyRegion)
plot(landmass, main  = "Coral Triangle Species records", col = "grey", border = "grey", axes = TRUE)
plot(absences, col ="gold", pch = 16, cex = 0.5, add = TRUE)
plot(presences_ct, col = "blue", pch = 16, cex = 0.5, add = TRUE)

presences_ct$PA <-1
absences$PA <- 0 
records <- rbind(presences_ct, absences)
plot(landmassCT, main = "Coral Reefs Species records", col = "grey", border = "grey", axes = TRUE)
plot(records, y = "PA", col = c("gold", "blue"), pch = 16, cex = 0.5, add= TRUE)

modelData <- generateModelData(records, envConditions = environmentalLayers, method = "blocks", kFolds = 10)
modelData$plotDatasets


model <- trainModel(modelData, algorithm = "brt")
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, main = "Predicted Species distribution", col = rev(topo.colors(100)))

model <- trainModel(modelData, algorithm = "maxent")

names(environmentalLayers)
names(environmentalLayers) <- c("Present_Benthic_OceanTemperature_LtMax","Present_Benthic_OceanTemperature_LtMin","Present_Benthic_pH_Mean","Present_Benthic_Salinity_Mean","Present_Surface_OceanTemperature_LtMax","Present_Surface_OceanTemperature_LtMin.tif")
plot(environmentalLayers, axes = TRUE)

model <- trainModel(modelData, algorithm = "maxent")
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, main  = "predicted Species distribution", col = rev(topo.colors(100)))

